# 0. Load Packages, Dataset and Basic Cleanup

In [ ]:
from google.colab import drive
import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
)
from sklearn.utils import resample

import warnings
warnings.filterwarnings("ignore")
plt.style.use("grayscale")

In [ ]:
drive.mount("/content/drive")
BASE_DIR = "./00_Labelling"
FIGURE_DIR = os.path.join(BASE_DIR, "figures_exploration")
os.makedirs(FIGURE_DIR, exist_ok=True)

In [ ]:
csv_path = os.path.join(BASE_DIR, "data/mental_health_unified_labels_final.csv")
df_raw = pd.read_csv(csv_path, index_col=0)
print(f"Loaded {len(df_raw)} rows, {df_raw.shape[1]} columns")
print("\nColumns:", list(df_raw.columns))
df_raw.head()

In [ ]:
# --- Column mapping ---
TEXT_COL  = "statement"
HUMAN_COL = "status"            # original Kaggle labels
AI_COL    = "u_label"           # gpt-4o-mini unified label

PROB_COLS = [
    "u_p_normal", "u_p_depression", "u_p_anxiety",
    "u_p_suicidal", "u_p_stress", "u_p_bipolar",
    "u_p_personality_disorder",
]

# Canonical display labels (used for plots and confusion matrices)
LABELS = [
    "Normal", "Depression", "Anxiety",
    "Suicidal", "Stress", "Bipolar",
    "Personality Disorder",
]

# Mapping from AI uppercase labels to canonical display labels
AI_TO_DISPLAY = {
    "NORMAL": "Normal",
    "DEPRESSION": "Depression",
    "ANXIETY": "Anxiety",
    "SUICIDAL": "Suicidal",
    "STRESS": "Stress",
    "BIPOLAR": "Bipolar",
    "PERSONALITY_DISORDER": "Personality Disorder",
}

# Mapping from human labels to canonical (fix case inconsistencies)
HUMAN_TO_DISPLAY = {
    "Anxiety": "Anxiety",
    "Normal": "Normal",
    "Depression": "Depression",
    "Suicidal": "Suicidal",
    "Stress": "Stress",
    "Bipolar": "Bipolar",
    "Personality disorder": "Personality Disorder",
}

SPECIAL_LABELS = {"OUT_OF_SCOPE", "INSUFFICIENT"}

In [ ]:
# --- Text Cleaning ---
def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+", "<URL>", text)
    text = re.sub(r"@\w+", "<USER>", text)
    text = re.sub(r"#\w+", "<HASHTAG>", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

print("Cleaning text...")
df = df_raw.copy()
df["text_clean"] = df[TEXT_COL].apply(clean_text)

# Map labels to canonical display format
df["human_label"] = df[HUMAN_COL].map(HUMAN_TO_DISPLAY)
df["ai_label"]    = df[AI_COL].map(AI_TO_DISPLAY)  # NaN for OUT_OF_SCOPE / INSUFFICIENT

print(f"Rows with valid human label: {df['human_label'].notna().sum()}")
print(f"Rows with valid AI label:    {df['ai_label'].notna().sum()}")
print(f"AI special labels:           {df[AI_COL].isin(SPECIAL_LABELS).sum()}")

In [ ]:
# --- Duplicate text analysis ---
text_counts = df["text_clean"].value_counts()
dup_texts = text_counts[text_counts > 1]

print(f"Total rows:           {len(df)}")
print(f"Unique texts:         {df['text_clean'].nunique()}")
print(f"Texts with duplicates: {len(dup_texts)}")
print(f"Rows in dup groups:   {df[df['text_clean'].isin(dup_texts.index)].shape[0]}")

# Consistency within duplicate groups
dup_mask = df["text_clean"].duplicated(keep=False)
dup_groups = df[dup_mask].copy()

grouped = (
    dup_groups
    .groupby("text_clean")
    .agg(
        n_rows         = (HUMAN_COL, "size"),
        n_human_labels = (HUMAN_COL, pd.Series.nunique),
        n_ai_labels    = (AI_COL, pd.Series.nunique),
    )
    .reset_index()
)
grouped["human_consistent"] = grouped["n_human_labels"] == 1
grouped["ai_consistent"]    = grouped["n_ai_labels"] == 1

n_groups = len(grouped)
print(f"\n=== Duplicate-group consistency ===")
print(f"Groups: {n_groups}")
print(f"Human-consistent: {grouped['human_consistent'].sum()} ({grouped['human_consistent'].mean():.1%})")
print(f"AI-consistent:    {grouped['ai_consistent'].sum()} ({grouped['ai_consistent'].mean():.1%})")

# 1. Scope & Sufficiency Gate Analysis

In [ ]:
# Which original labels get flagged as OUT_OF_SCOPE or INSUFFICIENT?
scope_df = df[df[AI_COL].isin(SPECIAL_LABELS)].copy()
print(f"Total special-label rows: {len(scope_df)}")
print(f"\nBreakdown by AI special label:")
print(scope_df[AI_COL].value_counts())

print(f"\nOriginal (human) labels of OUT_OF_SCOPE rows:")
print(scope_df[scope_df[AI_COL] == "OUT_OF_SCOPE"][HUMAN_COL].value_counts())

print(f"\nOriginal (human) labels of INSUFFICIENT rows:")
print(scope_df[scope_df[AI_COL] == "INSUFFICIENT"][HUMAN_COL].value_counts())

# Cross-tab
ct = pd.crosstab(scope_df[HUMAN_COL], scope_df[AI_COL], margins=True)
print("\nCross-tab: Original label x AI special label")
print(ct)

In [ ]:
# Text length comparison: special vs evaluable
df["text_len"] = df[TEXT_COL].str.len()
df["is_special"] = df[AI_COL].isin(SPECIAL_LABELS)

summary = df.groupby("is_special")["text_len"].describe()
print("Text length by scope status (is_special=True means OUT_OF_SCOPE/INSUFFICIENT):")
print(summary.round(1))

fig, ax = plt.subplots(figsize=(6, 3.5))
df.boxplot(column="text_len", by="is_special", ax=ax)
ax.set_xlabel("Flagged as special (OUT_OF_SCOPE / INSUFFICIENT)")
ax.set_ylabel("Text length (chars)")
ax.set_title("Text length: evaluable vs special")
plt.suptitle("")
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "text_length_by_scope.png"), dpi=300)
plt.show()

# 2. Label Distribution (Original vs AI)

In [ ]:
# Filter to rows where BOTH labels are valid 7-class
df_valid = df.dropna(subset=["human_label", "ai_label"]).copy()
print(f"Rows with valid human AND AI labels: {len(df_valid)}")

counts_h = df_valid["human_label"].value_counts().reindex(LABELS, fill_value=0)
counts_a = df_valid["ai_label"].value_counts().reindex(LABELS, fill_value=0)

print("\nOriginal label distribution:")
print(counts_h)
print("\nAI label distribution:")
print(counts_a)

# Side-by-side bar plot
x = np.arange(len(LABELS))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, counts_h, width, label="Original (Kaggle)", color="0.3", edgecolor="black")
ax.bar(x + width/2, counts_a, width, label="AI (gpt-4o-mini)", color="0.7", edgecolor="black")
ax.set_xticks(x)
ax.set_xticklabels(LABELS, rotation=45, ha="right")
ax.set_ylabel("Count")
ax.set_title("Label Distribution: Original vs AI")
ax.legend()
fig.patch.set_facecolor("white")
ax.set_facecolor("white")
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "label_distribution_original_vs_ai.png"), dpi=300)
plt.show()

# 3. Confusion Matrix, Classification Report & Cohen's Kappa

In [ ]:
def plot_confusion_matrix_counts_and_pct(
    y_row, y_col, labels, row_name="Rater A", col_name="Rater B",
    title="Confusion matrix", figsize=(7, 6), save_path=None, dpi=300,
):
    cm = confusion_matrix(y_row, y_col, labels=labels)
    total = cm.sum()
    annot = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            count = cm[i, j]
            pct = 100.0 * count / total if total > 0 else 0.0
            annot[i, j] = f"{count}\n({pct:.1f}%)"

    plt.figure(figsize=figsize)
    ax = sns.heatmap(
        cm, annot=annot, fmt="", cmap="Greys",
        xticklabels=labels, yticklabels=labels,
        vmin=0, cbar=False,
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    plt.xlabel(col_name)
    plt.ylabel(row_name)
    plt.title(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=dpi)
        print(f"Saved → {save_path}")
    plt.show()
    plt.close()
    return cm

In [ ]:
y_human = df_valid["human_label"]
y_ai    = df_valid["ai_label"]

cm = plot_confusion_matrix_counts_and_pct(
    y_row=y_human, y_col=y_ai, labels=LABELS,
    row_name="Original label",
    col_name="AI label (gpt-4o-mini)",
    title="Original vs AI Labels (Mental Health)",
    figsize=(8, 7),
    save_path=os.path.join(FIGURE_DIR, "cm_original_vs_ai.png"),
)

In [ ]:
# Classification report (treating original as ground truth)
print("=== Classification Report (Original = truth, AI = prediction) ===")
print(classification_report(y_human, y_ai, labels=LABELS, digits=4, zero_division=0))

# Overall agreement
n_total = len(df_valid)
n_agree = (y_human == y_ai).sum()
print(f"Exact agreement: {n_agree}/{n_total} ({n_agree/n_total:.1%})")

# Cohen's kappa
kappa = cohen_kappa_score(y_human, y_ai)
print(f"Cohen's kappa: {kappa:.4f}")

# Bootstrap CI for kappa
n_boot = 1000
rng = np.random.default_rng(2025)
indices = np.arange(n_total)
boot_kappa = []
for _ in range(n_boot):
    idx = rng.choice(indices, size=n_total, replace=True)
    boot_kappa.append(cohen_kappa_score(y_human.values[idx], y_ai.values[idx]))
boot_kappa = np.array(boot_kappa)
ci_lo, ci_hi = np.percentile(boot_kappa, [2.5, 97.5])
print(f"Bootstrap 95% CI for kappa: ({ci_lo:.4f}, {ci_hi:.4f})")

In [ ]:
# Per-class agreement rate
print("\n=== Per-class agreement rate ===")
for label in LABELS:
    mask = y_human == label
    n_class = mask.sum()
    n_agree_class = ((y_human == label) & (y_ai == label)).sum()
    rate = n_agree_class / n_class if n_class > 0 else 0
    print(f"  {label:25s}  {n_agree_class:>5}/{n_class:<5}  ({rate:.1%})")

# 4. Entropy from Soft Probability Labels

In [ ]:
# Ensure probability columns are numeric
for col in PROB_COLS:
    df_valid[col] = pd.to_numeric(df_valid[col], errors="coerce")

P = df_valid[PROB_COLS].values
K = P.shape[1]

# Normalize rows and compute raw Shannon entropy (nats)
eps = 1e-12
P_safe = np.clip(P, eps, 1.0)
P_safe = P_safe / P_safe.sum(axis=1, keepdims=True)
entropy = -np.sum(P_safe * np.log(P_safe), axis=1)  # raw entropy in nats; max = ln(K) ≈ 1.95
df_valid["ai_entropy"] = entropy

print(f"AI entropy (raw, nats; max = ln({K}) = {np.log(K):.4f}) statistics:")
print(df_valid["ai_entropy"].describe().round(4))

In [ ]:
# --- Panel (a): All Posts ---
ent = df_valid["ai_entropy"].to_numpy()
ent = np.clip(ent, 0, None)

is_zero = ent < 1e-6
print(f"N total: {len(ent)}")
print(f"N zero-entropy: {is_zero.sum()} ({is_zero.mean():.1%})")
print(f"N positive:     {(~is_zero).sum()}")

qs = np.quantile(ent, [0.25, 0.5, 0.75])

fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(ent, bins=40, edgecolor="black")
ax.set_xlabel("AI entropy (nats)")
ax.set_ylabel("Count")
ax.set_title("All Posts")
for v, lab in zip(qs, ["Q1", "Q2", "Q3"]):
    ax.axvline(v, ls="--", lw=1)
    ax.text(v, ax.get_ylim()[1]*0.9, lab, rotation=90, va="top", ha="right", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "entropy_all.png"), dpi=300, bbox_inches="tight")
plt.show()

# --- Panel (b): Entropy > 0 Only ---
ent_pos = ent[~is_zero]

fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(ent_pos, bins=40, edgecolor="black")
ax.set_xlabel("AI entropy (nats)")
ax.set_ylabel("Count")
ax.set_title("Entropy > 0 Only")
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "entropy_pos.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Entropy by original human label
entropy_by_status = (
    df_valid
    .groupby("human_label")["ai_entropy"]
    .agg(n="size", mean="mean", std="std", median="median",
         q25=lambda x: x.quantile(0.25), q75=lambda x: x.quantile(0.75))
    .reindex(LABELS)
    .round(4)
)
print("Entropy by original label:")
print(entropy_by_status)

# Box plot
fig, ax = plt.subplots(figsize=(8, 4))
order = entropy_by_status.sort_values("mean").index.tolist()
df_valid.boxplot(column="ai_entropy", by="human_label", ax=ax,
                 positions=range(len(order)),
                 widths=0.6)
ax.set_xticklabels(order, rotation=45, ha="right")
ax.set_xlabel("Original Label")
ax.set_ylabel("AI Entropy (nats)")
ax.set_title("AI Entropy by Original Label")
plt.suptitle("")
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "entropy_by_original_label.png"), dpi=300)
plt.show()

# 5. Entropy vs Human–AI Disagreement

In [ ]:
df_valid = df_valid.copy()
df_valid["disagree"] = (df_valid["human_label"] != df_valid["ai_label"]).astype(int)

# Disagreement by entropy quintile
df_valid["entropy_bin"] = pd.qcut(df_valid["ai_entropy"], q=5, duplicates="drop")
disagree_by_bin = (
    df_valid.groupby("entropy_bin", observed=False)["disagree"]
    .mean().reset_index()
)
print("Disagreement rate by entropy quintile:")
print(disagree_by_bin)

# Plot
labels_q = [f"Q{i+1}\n[{row.entropy_bin.left:.2f},{row.entropy_bin.right:.2f}]"
            for i, row in disagree_by_bin.iterrows()]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(len(disagree_by_bin)), disagree_by_bin["disagree"], edgecolor="black")
ax.set_xticks(range(len(disagree_by_bin)))
ax.set_xticklabels(labels_q)
ax.set_ylim(0, 1)
ax.set_ylabel("Disagreement rate")
ax.set_xlabel("AI entropy quintile (low → high)")
ax.set_title("Original–AI Disagreement vs AI Entropy")
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "disagreement_vs_entropy.png"), dpi=300)
plt.show()

In [ ]:
# Entropy distribution: agree vs disagree
entropy_by_agree = (
    df_valid.groupby("disagree")["ai_entropy"]
    .describe(percentiles=[0.25, 0.5, 0.75])
)
print("Entropy distribution by agreement:")
print(entropy_by_agree.round(4))

fig, ax = plt.subplots(figsize=(5, 3.5))
df_valid.boxplot(column="ai_entropy", by="disagree", ax=ax)
ax.set_xlabel("Disagreement (0 = agree, 1 = disagree)")
ax.set_ylabel("AI entropy")
ax.set_title("AI Entropy by Agreement Status")
plt.suptitle("")
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "entropy_by_agreement.png"), dpi=300)
plt.show()

# 6. High-Uncertainty Subset for Review

In [ ]:
threshold = df_valid["ai_entropy"].quantile(0.8)
high_unc = df_valid[df_valid["ai_entropy"] >= threshold]

print(f"Entropy threshold (80th pct): {threshold:.4f}")
print(f"High-uncertainty rows: {len(high_unc)}")
print(f"Disagreement rate (high-unc): {high_unc['disagree'].mean():.3f}")
print(f"Disagreement rate (all):      {df_valid['disagree'].mean():.3f}")

# Which classes are most affected?
print("\nHigh-uncertainty rows by original label:")
print(high_unc["human_label"].value_counts().reindex(LABELS, fill_value=0))

# 7. Logistic Regression: Predicting Disagreement

In [ ]:
import statsmodels.formula.api as smf
from sklearn.metrics import roc_auc_score

df_lr = df_valid.copy()
df_lr = df_lr.dropna(subset=["ai_entropy", "human_label"]).copy()

# Standardize entropy
ent_mean = df_lr["ai_entropy"].mean()
ent_std  = df_lr["ai_entropy"].std()
df_lr["entropy_std"] = (df_lr["ai_entropy"] - ent_mean) / ent_std
df_lr["human_label"] = df_lr["human_label"].astype("category")

y = df_lr["disagree"].values

# Model 1: disagree ~ entropy
model1 = smf.logit("disagree ~ entropy_std", data=df_lr).fit(disp=0)
print("=== Model 1: disagree ~ entropy_std ===")
print(model1.summary2())

or1 = np.exp(model1.params)
ci1 = np.exp(model1.conf_int())
print("\nOdds ratios:")
print(pd.DataFrame({"OR": or1, "CI_low": ci1[0], "CI_high": ci1[1]}).round(4))

auc1 = roc_auc_score(y, model1.predict(df_lr))
print(f"AUC: {auc1:.4f}")

# Model 2: disagree ~ entropy + original label
formula2 = "disagree ~ entropy_std + C(human_label, Treatment(reference='Normal'))"
model2 = smf.logit(formula2, data=df_lr).fit(disp=0)
print("\n=== Model 2: disagree ~ entropy_std + human_label ===")
print(model2.summary2())

or2 = np.exp(model2.params)
ci2 = np.exp(model2.conf_int())
print("\nOdds ratios:")
print(pd.DataFrame({"OR": or2, "CI_low": ci2[0], "CI_high": ci2[1]}).round(4))

auc2 = roc_auc_score(y, model2.predict(df_lr))
print(f"AUC: {auc2:.4f}")

# 8. AI Probability Calibration

In [ ]:
# Reliability diagram: Is the AI's max probability well-calibrated?
max_prob = P_safe.max(axis=1)
correct  = (df_valid["human_label"] == df_valid["ai_label"]).astype(int).values

n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

acc_per_bin = []
conf_per_bin = []
count_per_bin = []

for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask = (max_prob >= lo) & (max_prob < hi)
    if mask.sum() == 0:
        acc_per_bin.append(np.nan)
        conf_per_bin.append(np.nan)
        count_per_bin.append(0)
    else:
        acc_per_bin.append(correct[mask].mean())
        conf_per_bin.append(max_prob[mask].mean())
        count_per_bin.append(mask.sum())

acc_per_bin = np.array(acc_per_bin)
conf_per_bin = np.array(conf_per_bin)
count_per_bin = np.array(count_per_bin)

# Expected Calibration Error
valid_bins = count_per_bin > 0
ece = np.sum(count_per_bin[valid_bins] * np.abs(acc_per_bin[valid_bins] - conf_per_bin[valid_bins])) / count_per_bin.sum()
print(f"Expected Calibration Error (ECE): {ece:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Reliability diagram
ax = axes[0]
ax.bar(bin_centers, acc_per_bin, width=0.08, edgecolor="black", color="0.6", label="Accuracy")
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction correct")
ax.set_title(f"Reliability Diagram (ECE = {ece:.3f})")
ax.legend(loc="lower right")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Confidence histogram
ax2 = axes[1]
ax2.hist(max_prob, bins=40, edgecolor="black", color="0.6")
ax2.set_xlabel("AI max probability")
ax2.set_ylabel("Count")
ax2.set_title("Distribution of AI Confidence")

plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "calibration.png"), dpi=300)
plt.show()

# 9. Condition Strength vs Agreement

In [ ]:
# For each condition, check if strength (none/weak/clear) correlates with agreement
conditions = ["depression", "anxiety", "suicidal", "stress", "bipolar", "personality_disorder"]

strength_data = []
for cond in conditions:
    strength_col = f"u_{cond}_strength"
    if strength_col not in df_valid.columns:
        continue
    for strength in ["none", "weak", "clear"]:
        mask = df_valid[strength_col] == strength
        if mask.sum() == 0:
            continue
        n = mask.sum()
        disagree_rate = df_valid.loc[mask, "disagree"].mean()
        mean_entropy = df_valid.loc[mask, "ai_entropy"].mean()
        strength_data.append({
            "condition": cond, "strength": strength,
            "n": n, "disagree_rate": disagree_rate,
            "mean_entropy": mean_entropy,
        })

df_strength = pd.DataFrame(strength_data)
print("Disagreement rate and entropy by condition strength:")
print(df_strength.to_string(index=False))

# Pivot for heatmap
pivot_disagree = df_strength.pivot(index="condition", columns="strength", values="disagree_rate")
pivot_disagree = pivot_disagree[["none", "weak", "clear"]]

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(pivot_disagree, annot=True, fmt=".3f", cmap="Greys", ax=ax,
            vmin=0, vmax=pivot_disagree.max().max() * 1.1)
ax.set_title("Disagreement Rate by Condition × Strength")
ax.set_xlabel("Strength")
ax.set_ylabel("Condition")
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "strength_vs_disagreement.png"), dpi=300)
plt.show()

# 10. Disagreement Pattern Analysis

In [ ]:
# When AI disagrees, what does it predict instead?
disagree_df = df_valid[df_valid["disagree"] == 1].copy()
print(f"Total disagreement rows: {len(disagree_df)}")

# Transition matrix: Original → AI (for disagreements only)
trans = pd.crosstab(
    disagree_df["human_label"], disagree_df["ai_label"],
    margins=True
).reindex(index=LABELS + ["All"], columns=LABELS + ["All"], fill_value=0)
print("\nTransition matrix (Original → AI, disagreements only):")
print(trans)

# Heatmap of misclassification patterns
trans_core = trans.loc[LABELS, LABELS]
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(trans_core, annot=True, fmt="d", cmap="Greys", ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_xlabel("AI predicted label")
ax.set_ylabel("Original label")
ax.set_title("Misclassification Patterns (Disagreements Only)")
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "misclassification_patterns.png"), dpi=300)
plt.show()

In [ ]:
# Most common confusion pairs
pairs = []
for orig in LABELS:
    for ai in LABELS:
        if orig == ai:
            continue
        n = ((disagree_df["human_label"] == orig) & (disagree_df["ai_label"] == ai)).sum()
        if n > 0:
            pairs.append({"Original": orig, "AI_predicted": ai, "Count": n})

df_pairs = pd.DataFrame(pairs).sort_values("Count", ascending=False)
print("Top 15 confusion pairs:")
print(df_pairs.head(15).to_string(index=False))

# 11. Summary

In [ ]:
print("=" * 60)
print("SUMMARY — Original vs AI Label Analysis (Mental Health)")
print("=" * 60)
print(f"  Total rows:                  {len(df)}")
print(f"  Evaluable (both labels):     {len(df_valid)}")
print(f"  Special (OUT_OF_SCOPE/INSUFFICIENT): {df[AI_COL].isin(SPECIAL_LABELS).sum()}")
print(f"")
print(f"  Exact agreement:             {(df_valid['disagree']==0).sum()}/{len(df_valid)} "
      f"({(df_valid['disagree']==0).mean():.1%})")
print(f"  Cohen's kappa:               {kappa:.4f} (95% CI: {ci_lo:.4f}–{ci_hi:.4f})")
print(f"  Expected Calibration Error:  {ece:.4f}")
print(f"")
print(f"  High-uncertainty (top 20%):  {len(high_unc)} rows")
print(f"    Disagree rate (high-unc):  {high_unc['disagree'].mean():.3f}")
print(f"    Disagree rate (overall):   {df_valid['disagree'].mean():.3f}")
print(f"")
print(f"  Logistic regression AUC:")
print(f"    Model 1 (entropy only):    {auc1:.4f}")
print(f"    Model 2 (entropy + label): {auc2:.4f}")

# 12. Comorbidity Analysis — Prevalence & Co-occurrence

In [ ]:
# --- 12a. Number of co-occurring conditions ---
CONDITION_COLS = [
    "u_depression_present", "u_anxiety_present", "u_suicidal_present",
    "u_stress_present", "u_bipolar_present", "u_personality_disorder_present",
]
CONDITION_NAMES = ["Depression", "Anxiety", "Suicidal", "Stress", "Bipolar", "Pers. Disorder"]

# u_n_conditions is already in the dataset; verify it matches sum of _present cols
df_valid["n_conditions_check"] = df_valid[CONDITION_COLS].sum(axis=1).astype(int)
assert (df_valid["n_conditions_check"] == df_valid["u_n_conditions"]).all(), \
    "n_conditions mismatch!"
df_valid["n_cond"] = df_valid["u_n_conditions"].astype(int)

# Distribution
cond_dist = df_valid["n_cond"].value_counts().sort_index()
print("=== Distribution of number of co-occurring conditions ===")
for n, count in cond_dist.items():
    pct = count / len(df_valid) * 100
    print(f"  {n} conditions: {count:>6} ({pct:5.1f}%)")

print(f"\n  Mean: {df_valid['n_cond'].mean():.2f}")
print(f"  Rows with 0 conditions (Normal-inferred): {(df_valid['n_cond'] == 0).sum()}")
print(f"  Rows with 1 condition (single-label):     {(df_valid['n_cond'] == 1).sum()}")
print(f"  Rows with 2+ conditions (comorbid):       {(df_valid['n_cond'] >= 2).sum()} "
      f"({(df_valid['n_cond'] >= 2).mean():.1%})")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(cond_dist.index, cond_dist.values, edgecolor="black", color="0.6")
ax.set_xlabel("Number of conditions detected by AI")
ax.set_ylabel("Count")
ax.set_title("Distribution of Comorbid Conditions (AI assessment)")
for i, (n, count) in enumerate(cond_dist.items()):
    ax.text(n, count + 200, f"{count/len(df_valid)*100:.1f}%", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "comorbidity_n_conditions.png"), dpi=300)
plt.show()

In [ ]:
# --- 12b. Per-condition prevalence ---
print("=== Per-condition prevalence (AI assessment) ===")
for col, name in zip(CONDITION_COLS, CONDITION_NAMES):
    n_present = df_valid[col].sum()
    pct = n_present / len(df_valid) * 100
    print(f"  {name:20s}  {n_present:>6}  ({pct:5.1f}%)")

# Comorbidity rate by original human label
print("\n=== Comorbidity rate by original human label ===")
comorbid_by_label = (
    df_valid.groupby("human_label")
    .agg(
        n=("n_cond", "size"),
        mean_n_cond=("n_cond", "mean"),
        pct_comorbid=("n_cond", lambda x: (x >= 2).mean()),
        pct_zero=("n_cond", lambda x: (x == 0).mean()),
    )
    .reindex(LABELS)
    .round(3)
)
print(comorbid_by_label)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(LABELS))
ax.bar(x, comorbid_by_label["pct_comorbid"], edgecolor="black", color="0.5")
ax.set_xticks(x)
ax.set_xticklabels(LABELS, rotation=45, ha="right")
ax.set_ylabel("Proportion with 2+ conditions")
ax.set_title("Comorbidity Rate by Original Human Label")
ax.set_ylim(0, 1)
for i, v in enumerate(comorbid_by_label["pct_comorbid"]):
    ax.text(i, v + 0.02, f"{v:.1%}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "comorbidity_rate_by_label.png"), dpi=300)
plt.show()

In [ ]:
# --- 12c. Condition co-occurrence matrix ---
# Binary matrix: which conditions are present together
present_matrix = df_valid[CONDITION_COLS].values.astype(int)

# Co-occurrence: how often condition i and condition j are both present
n_conditions = len(CONDITION_COLS)
cooccur = np.zeros((n_conditions, n_conditions), dtype=int)
for i in range(n_conditions):
    for j in range(n_conditions):
        cooccur[i, j] = ((present_matrix[:, i] == 1) & (present_matrix[:, j] == 1)).sum()

df_cooccur = pd.DataFrame(cooccur, index=CONDITION_NAMES, columns=CONDITION_NAMES)
print("=== Condition co-occurrence matrix (counts) ===")
print(df_cooccur)

# Conditional probability: P(j present | i present)
print("\n=== Conditional co-occurrence: P(column | row present) ===")
diag = np.diag(cooccur).astype(float)
diag[diag == 0] = 1  # avoid division by zero
cond_prob = cooccur / diag[:, None]
df_cond_prob = pd.DataFrame(cond_prob, index=CONDITION_NAMES, columns=CONDITION_NAMES).round(3)
print(df_cond_prob)

# Heatmap of conditional co-occurrence (mask diagonal)
mask = np.eye(n_conditions, dtype=bool)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    df_cond_prob, annot=True, fmt=".2f", cmap="Greys",
    mask=mask, vmin=0, vmax=0.6, ax=ax,
)
ax.set_title("P(column condition | row condition present)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "comorbidity_cooccurrence_heatmap.png"), dpi=300)
plt.show()

# 13. Comorbidity Drives Label Confusion

In [ ]:
# --- 13a. Comorbidity rate: agree vs disagree ---
df_valid["is_comorbid"] = (df_valid["n_cond"] >= 2).astype(int)

agree_mask = df_valid["disagree"] == 0
disagree_mask = df_valid["disagree"] == 1

comorbid_agree = df_valid.loc[agree_mask, "is_comorbid"].mean()
comorbid_disagree = df_valid.loc[disagree_mask, "is_comorbid"].mean()

print("=== Comorbidity rate by agreement status ===")
print(f"  Agree samples:    {comorbid_agree:.1%} comorbid  (n={agree_mask.sum()})")
print(f"  Disagree samples: {comorbid_disagree:.1%} comorbid  (n={disagree_mask.sum()})")
print(f"  Overall:          {df_valid['is_comorbid'].mean():.1%} comorbid")

# Mean number of conditions
print(f"\n  Mean conditions (agree):    {df_valid.loc[agree_mask, 'n_cond'].mean():.2f}")
print(f"  Mean conditions (disagree): {df_valid.loc[disagree_mask, 'n_cond'].mean():.2f}")

# Disagreement rate by n_conditions
print("\n=== Disagreement rate by number of conditions ===")
disagree_by_ncond = (
    df_valid.groupby("n_cond")["disagree"]
    .agg(n="size", disagree_rate="mean")
    .reset_index()
)
print(disagree_by_ncond.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(disagree_by_ncond["n_cond"], disagree_by_ncond["disagree_rate"],
       edgecolor="black", color="0.5")
ax.set_xlabel("Number of conditions (AI)")
ax.set_ylabel("Disagreement rate")
ax.set_title("Human–AI Disagreement vs Number of Comorbid Conditions")
ax.set_ylim(0, 1)
for _, row in disagree_by_ncond.iterrows():
    ax.text(row["n_cond"], row["disagree_rate"] + 0.02,
            f'{row["disagree_rate"]:.1%}', ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "disagreement_vs_n_conditions.png"), dpi=300)
plt.show()

In [ ]:
# --- 13a-ii. Statistical test: comorbidity → disagreement ---
from scipy.stats import chi2_contingency

# 2x2 contingency table: comorbid (yes/no) x disagree (yes/no)
ct = pd.crosstab(df_valid["is_comorbid"], df_valid["disagree"])
print("=== Contingency table: Comorbid × Disagree ===")
ct.index = ["Non-comorbid", "Comorbid"]
ct.columns = ["Agree", "Disagree"]
print(ct)

chi2, p, dof, expected = chi2_contingency(ct)
print(f"\nChi-squared = {chi2:.2f}, df = {dof}, p = {p:.2e}")

# Odds ratio with 95% CI
a, b = ct.iloc[1, 1], ct.iloc[1, 0]  # comorbid: disagree, agree
c, d = ct.iloc[0, 1], ct.iloc[0, 0]  # non-comorbid: disagree, agree
OR = (a * d) / (b * c)
log_OR = np.log(OR)
SE_log_OR = np.sqrt(1/a + 1/b + 1/c + 1/d)
ci_lo_or = np.exp(log_OR - 1.96 * SE_log_OR)
ci_hi_or = np.exp(log_OR + 1.96 * SE_log_OR)

print(f"\nOdds Ratio (comorbid → disagree): {OR:.3f} (95% CI: {ci_lo_or:.3f}–{ci_hi_or:.3f})")
print(f"Interpretation: Comorbid samples have {OR:.1f}x the odds of human–AI "
      f"disagreement compared to non-comorbid samples.")

In [ ]:
# --- 13a-iii. Panel C agreement subset: comorbidity profile ---
# Panel C uses only rows where human_label == ai_label (agreement subset).
# Show how filtering to agreement depletes comorbid cases.

agree_sub = df_valid[df_valid["disagree"] == 0].copy()
disagree_sub = df_valid[df_valid["disagree"] == 1].copy()

print("=" * 60)
print("PANEL C AGREEMENT SUBSET — COMORBIDITY PROFILE")
print("=" * 60)

# 1. Basic sizes
print(f"\nFull dataset:       N = {len(df_valid):,}")
print(f"Agreement subset:   N = {len(agree_sub):,} ({len(agree_sub)/len(df_valid):.1%})")
print(f"Disagreement subset: N = {len(disagree_sub):,} ({len(disagree_sub)/len(df_valid):.1%})")

# 2. Comorbidity rates
comorbid_full = df_valid["is_comorbid"].mean()
comorbid_agree = agree_sub["is_comorbid"].mean()
comorbid_disagree = disagree_sub["is_comorbid"].mean()

print(f"\n--- Comorbidity rates ---")
print(f"  Full dataset:       {comorbid_full:.1%}")
print(f"  Agreement subset:   {comorbid_agree:.1%}  (← Panel C training data)")
print(f"  Disagreement subset: {comorbid_disagree:.1%}  (excluded from Panel C)")
print(f"  Relative depletion:  {(comorbid_full - comorbid_agree)/comorbid_full:.1%} fewer comorbid cases in Panel C")

# 3. Mean conditions
print(f"\n--- Mean number of conditions ---")
print(f"  Full dataset:       {df_valid['n_cond'].mean():.2f}")
print(f"  Agreement subset:   {agree_sub['n_cond'].mean():.2f}")
print(f"  Disagreement subset: {disagree_sub['n_cond'].mean():.2f}")

# 4. Class distribution shift
print(f"\n--- Class distribution: Full vs Agreement subset ---")
full_dist = df_valid["human_label"].value_counts(normalize=True).sort_index()
agree_dist = agree_sub["human_label"].value_counts(normalize=True).sort_index()

dist_compare = pd.DataFrame({
    "Full (%)": (full_dist * 100).round(1),
    "Panel C (%)": (agree_dist * 100).round(1),
    "Δ (pp)": ((agree_dist - full_dist) * 100).round(1)
})
print(dist_compare.to_string())

# 5. Per-class comorbidity rates: agreement vs disagreement
print(f"\n--- Per-class comorbidity rate: agree vs disagree ---")
per_class_stats = []
for label in sorted(df_valid["human_label"].unique()):
    mask_label = df_valid["human_label"] == label
    agree_comorbid = df_valid.loc[mask_label & (df_valid["disagree"] == 0), "is_comorbid"].mean()
    disagree_comorbid = df_valid.loc[mask_label & (df_valid["disagree"] == 1), "is_comorbid"].mean()
    n_agree = (mask_label & (df_valid["disagree"] == 0)).sum()
    n_disagree = (mask_label & (df_valid["disagree"] == 1)).sum()
    per_class_stats.append({
        "Class": label,
        "n_agree": n_agree,
        "n_disagree": n_disagree,
        "comorbid_agree": f"{agree_comorbid:.1%}" if n_agree > 0 else "—",
        "comorbid_disagree": f"{disagree_comorbid:.1%}" if n_disagree > 0 else "—"
    })
per_class_df = pd.DataFrame(per_class_stats)
print(per_class_df.to_string(index=False))

# 6. Visualization: side-by-side comorbidity distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) n_conditions distribution
for subset, label, color in [(agree_sub, "Agreement (Panel C)", "steelblue"),
                              (disagree_sub, "Disagreement", "salmon")]:
    counts = subset["n_cond"].value_counts(normalize=True).sort_index()
    axes[0].plot(counts.index, counts.values, "o-", label=label, color=color)
axes[0].set_xlabel("Number of conditions")
axes[0].set_ylabel("Proportion")
axes[0].set_title("Condition count distribution")
axes[0].legend()

# (b) Class distribution
x_pos = np.arange(len(full_dist))
width = 0.35
axes[1].bar(x_pos - width/2, full_dist.values * 100, width, label="Full dataset", color="0.6")
axes[1].bar(x_pos + width/2, agree_dist.values * 100, width, label="Panel C (agree)", color="steelblue")
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(full_dist.index, rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Percentage (%)")
axes[1].set_title("Class distribution shift")
axes[1].legend()

# (c) Per-condition prevalence: agree vs full
cond_prev_full = df_valid[CONDITION_COLS].mean()
cond_prev_agree = agree_sub[CONDITION_COLS].mean()
x_pos_c = np.arange(len(CONDITION_NAMES))
axes[2].bar(x_pos_c - width/2, cond_prev_full.values * 100, width, label="Full", color="0.6")
axes[2].bar(x_pos_c + width/2, cond_prev_agree.values * 100, width, label="Panel C", color="steelblue")
axes[2].set_xticks(x_pos_c)
axes[2].set_xticklabels(CONDITION_NAMES, rotation=45, ha="right", fontsize=8)
axes[2].set_ylabel("Prevalence (%)")
axes[2].set_title("Per-condition prevalence shift")
axes[2].legend()

plt.suptitle("Panel C Agreement Subset: Comorbidity Depletion Effects", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "panel_c_comorbidity_depletion.png"), dpi=300, bbox_inches="tight")
plt.show()

# 7. Key takeaway
print("\n" + "=" * 60)
print("KEY IMPLICATION FOR PANEL C")
print("=" * 60)
print("""
The agreement subset (Panel C) is systematically depleted of comorbid cases:
  • Comorbid samples disagree more → filtered out by agreement criterion
  • This makes Panel C's training data 'easier' (less ambiguous)
  • Classes most affected by comorbidity (e.g., Depression, Suicidal) lose
    disproportionately more samples in Panel C
  • Panel C performance metrics may appear inflated because the hardest
    cases (comorbid, high-entropy) are excluded from its training data
""")

In [ ]:
# --- 13b. Top confusion pairs explained by comorbidity ---
# For the top confusion pairs from Section 10, check what fraction are comorbid
TOP_PAIRS = [
    ("Depression", "Suicidal"),
    ("Depression", "Normal"),
    ("Depression", "Anxiety"),
    ("Suicidal", "Depression"),
    ("Suicidal", "Normal"),
    ("Stress", "Anxiety"),
    ("Bipolar", "Normal"),
    ("Stress", "Normal"),
    ("Bipolar", "Anxiety"),
    ("Personality Disorder", "Normal"),
]

print("=== Top confusion pairs: comorbidity explanation ===")
print(f"{'Original':<22} {'AI predicted':<22} {'N':>5} {'% comorbid':>10} {'Mean n_cond':>11}")
print("-" * 75)
for orig, ai_pred in TOP_PAIRS:
    mask = (df_valid["human_label"] == orig) & (df_valid["ai_label"] == ai_pred)
    n = mask.sum()
    if n == 0:
        continue
    pct_comorbid = df_valid.loc[mask, "is_comorbid"].mean()
    mean_nc = df_valid.loc[mask, "n_cond"].mean()
    print(f"  {orig:<20} {ai_pred:<20} {n:>5} {pct_comorbid:>9.1%} {mean_nc:>11.2f}")

# Compare with matched-label (agree) rows
print(f"\n  {'AGREE samples':<20} {'(all)':<20} {agree_mask.sum():>5} "
      f"{df_valid.loc[agree_mask, 'is_comorbid'].mean():>9.1%} "
      f"{df_valid.loc[agree_mask, 'n_cond'].mean():>11.2f}")

In [ ]:
# --- 13c. Confusion matrix: comorbid vs non-comorbid ---
# Split the data and show confusion matrices side by side

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for idx, (label, mask) in enumerate([
    ("Non-comorbid (0–1 conditions)", df_valid["is_comorbid"] == 0),
    ("Comorbid (2+ conditions)", df_valid["is_comorbid"] == 1),
]):
    subset = df_valid[mask]
    cm = confusion_matrix(subset["human_label"], subset["ai_label"], labels=LABELS)
    # Normalize by row (original label)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_pct = np.nan_to_num(cm_pct)

    annot = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f"{cm_pct[i,j]:.0%}" if cm_pct[i,j] >= 0.01 else ""

    ax = axes[idx]
    sns.heatmap(cm_pct, annot=annot, fmt="", cmap="Greys",
                xticklabels=LABELS, yticklabels=LABELS,
                vmin=0, vmax=1, ax=ax, cbar=False)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_xlabel("AI label")
    ax.set_ylabel("Original label")
    n_agree = (subset["human_label"] == subset["ai_label"]).sum()
    agree_pct = n_agree / len(subset) * 100
    ax.set_title(f"{label}\n(n={len(subset):,}, agreement={agree_pct:.1f}%)")

plt.suptitle("Row-normalized Confusion: Non-comorbid vs Comorbid", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "cm_comorbid_vs_noncomorbid.png"), dpi=300,
            bbox_inches="tight")
plt.show()

# 14. Comorbidity Motivates Soft Labels & Multi-Task Learning

In [ ]:
# --- 14a. Comorbidity → Soft probability spread ---
# Comorbid samples should have probability mass across multiple conditions
# Non-comorbid samples should have concentrated (one-hot-like) probability

# Metrics: entropy, max prob, sum of clinical probs
stats_by_comorbid = (
    df_valid.groupby("is_comorbid")
    .agg(
        n=("ai_entropy", "size"),
        mean_entropy=("ai_entropy", "mean"),
        median_entropy=("ai_entropy", "median"),
        mean_max_prob=("u_max_clinical_prob", "mean"),
        mean_sum_clinical=("u_sum_clinical_prob", "mean"),
        disagree_rate=("disagree", "mean"),
    )
    .round(4)
)
stats_by_comorbid.index = ["Non-comorbid (0-1)", "Comorbid (2+)"]
print("=== Probability spread: comorbid vs non-comorbid ===")
print(stats_by_comorbid.T)

# Same breakdown by n_conditions
print("\n=== Detailed breakdown by number of conditions ===")
stats_by_n = (
    df_valid.groupby("n_cond")
    .agg(
        n=("ai_entropy", "size"),
        mean_entropy=("ai_entropy", "mean"),
        mean_max_prob=("u_max_clinical_prob", "mean"),
        mean_sum_clinical=("u_sum_clinical_prob", "mean"),
        disagree_rate=("disagree", "mean"),
    )
    .round(4)
)
print(stats_by_n)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Entropy by n_conditions
axes[0].bar(stats_by_n.index, stats_by_n["mean_entropy"], edgecolor="black", color="0.5")
axes[0].set_xlabel("Number of conditions")
axes[0].set_ylabel("Mean AI entropy")
axes[0].set_title("Entropy increases with comorbidity")

# Max prob by n_conditions
axes[1].bar(stats_by_n.index, stats_by_n["mean_max_prob"], edgecolor="black", color="0.5")
axes[1].set_xlabel("Number of conditions")
axes[1].set_ylabel("Mean max clinical prob")
axes[1].set_title("Max prob decreases with comorbidity")

# Sum clinical prob by n_conditions
axes[2].bar(stats_by_n.index, stats_by_n["mean_sum_clinical"], edgecolor="black", color="0.5")
axes[2].set_xlabel("Number of conditions")
axes[2].set_ylabel("Mean sum of clinical probs")
axes[2].set_title("Total clinical signal grows with comorbidity")

plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "comorbidity_soft_label_motivation.png"), dpi=300)
plt.show()

In [ ]:
# --- 14b. Information lost by hard labels: comorbid examples ---
# Show a few concrete examples where hard label throws away comorbidity signal
print("=== Examples: comorbid samples where hard label loses information ===\n")

# Map prob columns to display names (all 7 including Normal)
PROB_DISPLAY = ["Normal", "Depression", "Anxiety", "Suicidal", "Stress", "Bipolar", "Pers. Disorder"]

# Comorbid, high entropy, disagreement
examples = df_valid[
    (df_valid["n_cond"] >= 2)
    & (df_valid["ai_entropy"] > 0.58)  # ~0.3 on normalized scale × ln(7)
    & (df_valid["disagree"] == 1)
].sort_values("ai_entropy", ascending=False).head(8)

for _, row in examples.iterrows():
    print(f"  Text:     {row[TEXT_COL][:100]}...")
    print(f"  Original: {row['human_label']}")
    print(f"  AI label: {row['ai_label']}")
    probs = {PROB_DISPLAY[i]: row[PROB_COLS[i]] for i in range(len(PROB_COLS))}
    nonzero = {k: f"{v:.2f}" for k, v in probs.items() if v > 0.01}
    print(f"  Probs:    {nonzero}")
    present = [CONDITION_NAMES[i] for i, col in enumerate(CONDITION_COLS) if row[col]]
    print(f"  Present:  {present}  (n_cond={row['n_cond']:.0f})")
    print(f"  Entropy:  {row['ai_entropy']:.3f}")
    print()

In [ ]:
# --- 14c. Multi-task justification: condition strength correlations ---
# If conditions co-occur and share textual signals, multi-task learning
# can leverage shared representations

STRENGTH_COLS = [
    "u_depression_strength", "u_anxiety_strength", "u_suicidal_strength",
    "u_stress_strength", "u_bipolar_strength", "u_personality_disorder_strength",
]
STRENGTH_MAP = {"none": 0, "weak": 1, "clear": 2}

# Encode strengths numerically
strength_numeric = df_valid[STRENGTH_COLS].copy()
for col in STRENGTH_COLS:
    strength_numeric[col] = strength_numeric[col].map(STRENGTH_MAP).fillna(0).astype(int)
strength_numeric.columns = CONDITION_NAMES

# Correlation matrix of condition strengths
corr = strength_numeric.corr()
print("=== Condition strength correlation matrix ===")
print(corr.round(3))

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Greys",
            mask=mask, vmin=-0.1, vmax=0.5, ax=ax)
ax.set_title("Correlation of Condition Strengths\n(none=0, weak=1, clear=2)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "condition_strength_correlation.png"), dpi=300)
plt.show()

# Highlight strongest correlations
print("\n=== Top condition-pair correlations ===")
corr_pairs = []
for i in range(len(CONDITION_NAMES)):
    for j in range(i+1, len(CONDITION_NAMES)):
        corr_pairs.append({
            "Condition A": CONDITION_NAMES[i],
            "Condition B": CONDITION_NAMES[j],
            "Correlation": corr.iloc[i, j],
        })
df_corr_pairs = pd.DataFrame(corr_pairs).sort_values("Correlation", ascending=False)
print(df_corr_pairs.head(10).to_string(index=False))

In [ ]:
# --- 14d. Multilabel distribution ---
# u_multilabel contains the multi-label assignments (e.g., "DEPRESSION+ANXIETY")
print("=== Multi-label distribution ===")
ml_counts = df_valid["u_multilabel"].value_counts()
print(f"Unique multi-label patterns: {len(ml_counts)}")
print(f"\nTop 20 multi-label patterns:")
print(ml_counts.head(20).to_string())

# Single vs multi-label
is_multi = df_valid["u_multilabel"].str.contains(r"\+", na=False)
print(f"\n  Single-label:  {(~is_multi).sum()} ({(~is_multi).mean():.1%})")
print(f"  Multi-label:   {is_multi.sum()} ({is_multi.mean():.1%})")

# Disagreement rate for multi-label vs single-label
print(f"\n  Disagree rate (single-label): {df_valid.loc[~is_multi, 'disagree'].mean():.1%}")
print(f"  Disagree rate (multi-label):  {df_valid.loc[is_multi, 'disagree'].mean():.1%}")

# 15. Comorbidity Summary

In [ ]:
print("=" * 60)
print("COMORBIDITY ANALYSIS SUMMARY")
print("=" * 60)

n_total = len(df_valid)
n_comorbid = df_valid["is_comorbid"].sum()
n_multi = is_multi.sum()

print(f"\n--- Prevalence ---")
print(f"  Total samples:               {n_total}")
print(f"  Comorbid (2+ conditions):    {n_comorbid} ({n_comorbid/n_total:.1%})")
print(f"  Multi-label patterns:        {n_multi} ({n_multi/n_total:.1%})")
print(f"  Mean conditions per sample:  {df_valid['n_cond'].mean():.2f}")

print(f"\n--- Comorbidity → Label Confusion ---")
print(f"  Disagree rate (non-comorbid):  {df_valid.loc[df_valid['is_comorbid']==0, 'disagree'].mean():.1%}")
print(f"  Disagree rate (comorbid):      {df_valid.loc[df_valid['is_comorbid']==1, 'disagree'].mean():.1%}")
print(f"  → Comorbid samples are ~{df_valid.loc[df_valid['is_comorbid']==1, 'disagree'].mean() / df_valid.loc[df_valid['is_comorbid']==0, 'disagree'].mean():.1f}x more likely to have human–AI disagreement")

print(f"\n--- Implications for Training Strategy ---")
print(f"  1. SOFT LABELS: Comorbid samples have mean entropy "
      f"{df_valid.loc[df_valid['is_comorbid']==1, 'ai_entropy'].mean():.3f} "
      f"vs {df_valid.loc[df_valid['is_comorbid']==0, 'ai_entropy'].mean():.3f} "
      f"for non-comorbid.")
print(f"     Hard one-hot labels discard this uncertainty → soft probability")
print(f"     distributions preserve clinically meaningful comorbidity signal.")
print(f"  2. MULTI-TASK: Conditions co-occur (e.g., Depression–Suicidal,")
print(f"     Depression–Anxiety) → shared textual features can be learned")
print(f"     jointly via multi-task heads for per-condition strength.")
print(f"  3. CONFUSION MATRIX: Top confusion pairs (Depression↔Suicidal,")
print(f"     Depression↔Anxiety) align with highest comorbidity rates,")
print(f"     suggesting disagreement reflects genuine clinical overlap,")
print(f"     not annotation noise.")